# AMC page structure
**individual movies:**
https://www.amctheatres.com/movies/[movie-name]/showtimes?date=[date]?theatre=[theatre-name]
> example:
> https://www.amctheatres.com/movies/disclosure-day-77140/showtimes?date=2026-07-22&theatre=amc-alderwood-mall-16

**all movies:**
https://www.amctheatres.com/movie-theatres/[general-location]/[theatre-name]/showtimes?date=[date]
> example:
> https://www.amctheatres.com/movie-theatres/seattle-tacoma/amc-woodinville-12/showtimes?date=2026-07-22

In [1]:
import json
import textwrap

import requests
from bs4 import BeautifulSoup
from IPython.display import HTML


In [ ]:
# original AMC graphql query

amc_query = """query TheatreSelectModalQuery($query: String!, $resultCount: Int = 10) {
  viewer {
    location(query: $query, first: $resultCount) {
      edges {
        node {
          ...TheatreSelectResult
        }
      }
    }
  }
}

fragment TheatreSelectTheatre on Theatre {
  name
  theatreId
  slug
  marketSlug
  isSelected
  longName
  secondaryLongName
  addressLine1
  addressLine2
  city
  state
  stateCode
  postalCode
  latitude
  longitude
  distance
  brand
  attributes(type: FILTERS, first: 4) {
    edges {
      node {
        name
      }
    }
  }
}

fragment TheatreSelectResult on Location {
  title
  theatres {
    edges {
      node {
        ...TheatreSelectTheatre
      }
    }
  }
}"""

search_query = "new york"
variables = f'{{"query": "{search_query}"}}'


In [2]:
class TheaterLookupQueryHandler:
    def __init__(
        self,
        default_result_count: int = 10,
        location_fields: list[str] | None = None,
        theatre_fields: list[str] | None = None,
        attributes_count: int | None = None,
        attributes_fields: None | list[str] = None,
    ) -> None:
        if location_fields is None:
            location_fields = ["title", "latitude", "longitude"]

        if theatre_fields is None:
            theatre_fields = [
                "name",
                "theatreId",
                "slug",
                "marketSlug",
                "isSelected",
                "longName",
                "secondaryLongName",
                "addressLine1",
                "addressLine2",
                "city",
                "state",
                "stateCode",
                "postalCode",
                "latitude",
                "longitude",
                "distance",
                "brand",
                "attributes",
            ]

        self.default_result_count = default_result_count
        self.location_fields = location_fields
        self.theatre_fields = theatre_fields

        if "attributes" in self.theatre_fields:
            if attributes_fields is None:
                attributes_fields = ["name", "id"]

            if attributes_count is None:
                attributes_count = 4

            self.attributes_count = attributes_count
            self.attributes_fields = attributes_fields

    def build_query(self) -> str:
        base_dedent_value = 4*3

        if "attributes" in self.theatre_fields:
            self.attributes_block = textwrap.indent(textwrap.dedent(f"""
                attributes(type: FILTERS, first: {self.attributes_count}) {{
                  edges {{
                    node {{
                      {('\n'+' '*(base_dedent_value+2+2+3*2)).join(self.attributes_fields)}
                    }}
                  }}
                }}"""
            ), " "*(base_dedent_value+2))

        self.query = textwrap.dedent(f"""
            query TheaterLookupQuery($query: String!, $resultCount: Int = {self.default_result_count}) {{
              viewer {{
                location(query: $query, first: $resultCount) {{
                  edges {{
                    node {{
                      ...TheatreSelectResult
                    }}
                  }}
                }}
              }}
            }}

            fragment TheatreSelectResult on Location {{
              {("\n"+" "*(base_dedent_value+2)).join(self.location_fields)}
              theatres {{
                edges {{
                  node {{
                    ...TheatreSelectTheatre
                  }}
                }}
              }}
            }}

            fragment TheatreSelectTheatre on Theatre {{
              {("\n"+" "*(base_dedent_value+2)).join(
                  [
                      p if p != "attributes" else self.attributes_block
                      for p in self.theatre_fields
                  ]
              )}
            }}""")

        return self.query

    def send_query(self, search_query: str, result_count: int | None = None):
        self.result = requests.post(
            "https://graph.amctheatres.com/",
            json={
                "query": self.build_query(),
                "variables": json.dumps(
                    {
                        "query": search_query,
                        "resultCount": result_count or self.default_result_count,
                    }
                ),
            },
        )
        print(json.dumps(self.result.json(), indent=2))
        return self.result.json()

In [7]:
query_handler = TheaterLookupQueryHandler(attributes_fields=["name", "id", "description"])

In [8]:
query_handler.build_query()

'\nquery TheaterLookupQuery($query: String!, $resultCount: Int = 10) {\n  viewer {\n    location(query: $query, first: $resultCount) {\n      edges {\n        node {\n          ...TheatreSelectResult\n        }\n      }\n    }\n  }\n}\n\nfragment TheatreSelectResult on Location {\n  title\n  latitude\n  longitude\n  theatres {\n    edges {\n      node {\n        ...TheatreSelectTheatre\n      }\n    }\n  }\n}\n\nfragment TheatreSelectTheatre on Theatre {\n  name\n  theatreId\n  slug\n  marketSlug\n  isSelected\n  longName\n  secondaryLongName\n  addressLine1\n  addressLine2\n  city\n  state\n  stateCode\n  postalCode\n  latitude\n  longitude\n  distance\n  brand\n\n  attributes(type: FILTERS, first: 4) {\n    edges {\n      node {\n        name\n        id\n        description\n      }\n    }\n  }\n}'

In [12]:
print(query_handler.query)


query TheaterLookupQuery($query: String!, $resultCount: Int = 10) {
  viewer {
    location(query: $query, first: $resultCount) {
      edges {
        node {
          ...TheatreSelectResult
        }
      }
    }
  }
}

fragment TheatreSelectResult on Location {
  title
  latitude
  longitude
  theatres {
    edges {
      node {
        ...TheatreSelectTheatre
      }
    }
  }
}

fragment TheatreSelectTheatre on Theatre {
  name
  theatreId
  slug
  marketSlug
  isSelected
  longName
  secondaryLongName
  addressLine1
  addressLine2
  city
  state
  stateCode
  postalCode
  latitude
  longitude
  distance
  brand

  attributes(type: FILTERS, first: 4) {
    edges {
      node {
        name
        id
        description
      }
    }
  }
}


In [13]:
query_handler.send_query("new york")

{
  "data": {
    "viewer": {
      "location": {
        "edges": [
          {
            "node": {
              "title": "New York, NY",
              "latitude": 40.71304703,
              "longitude": -74.00723267,
              "theatres": {
                "edges": [
                  {
                    "node": {
                      "name": "AMC Village 7",
                      "theatreId": 2110,
                      "slug": "amc-village-7",
                      "marketSlug": "new-york-city",
                      "isSelected": false,
                      "longName": "AMC Village 7",
                      "secondaryLongName": null,
                      "addressLine1": "66 Third Ave (11th St.)",
                      "addressLine2": null,
                      "city": "New York",
                      "state": "New York",
                      "stateCode": "NY",
                      "postalCode": "10003",
                      "latitude": 40.73141,
                  

{'data': {'viewer': {'location': {'edges': [{'node': {'title': 'New York, NY',
       'latitude': 40.71304703,
       'longitude': -74.00723267,
       'theatres': {'edges': [{'node': {'name': 'AMC Village 7',
           'theatreId': 2110,
           'slug': 'amc-village-7',
           'marketSlug': 'new-york-city',
           'isSelected': False,
           'longName': 'AMC Village 7',
           'secondaryLongName': None,
           'addressLine1': '66 Third Ave (11th St.)',
           'addressLine2': None,
           'city': 'New York',
           'state': 'New York',
           'stateCode': 'NY',
           'postalCode': '10003',
           'latitude': 40.73141,
           'longitude': -73.988568,
           'distance': 1.6,
           'brand': 'AMC',
           'attributes': {'edges': [{'node': {'name': 'Discount Matinees',
               'id': 'QXR0cmlidXRlOmlkfDI5MDpkaXNjb3VudG1hdGluZWVz',
               'description': 'Save 25% off the evening ticket prices every day!'}},
     

___

In [ ]:
with open("query.graphql", "r") as query:
    amc_query = query.read()

print(f"sending request... (Query Name: {amc_query.split('(')[0].split(' ')[1]})\n")
print(
    json.dumps(
        requests.post(
            "https://graph.amctheatres.com/", json={"query": amc_query}
        ).json(),
        indent=2,
    )
)

sending request... (Query Name: TheatreSelectModalQuery)

{
  "errors": [
    {
      "message": "Cannot query field \"state\" on type \"Attribute\".",
      "locations": [
        {
          "line": 36,
          "column": 9
        }
      ],
      "extensions": {
        "code": "GRAPHQL_VALIDATION_FAILED"
      }
    }
  ]
}


In [ ]:
print(f"sending request... (query: {search_query})\n")
response = requests.post(
    "https://graph.amctheatres.com/",
    json={"query": amc_query, "variables": json.loads(variables)},
)
print("data recieved")

NameError: name 'search_query' is not defined

In [3]:
def construct_url(date, theatre_name, movie_name=None):
    if movie_name:
        return f"https://amctheatres.com/movies/{movie_name}/showtimes?date={date}&theatre={theatre_name}"

In [ ]:
def render_url(url):
    html = requests.get(url).text
    return HTML(html)


# render_url("https://www.amctheatres.com/movies/disclosure-day-77140/showtimes?date=2026-07-22")

In [1]:
# render_url("https://www.amctheatres.com/movies/my-neighbor-totoro-studio-ghibli-fest-2026-83590/showtimes?theatre=amc-woodinville-12&date=2026-07-19")

In [2]:
# render_url("https://www.amctheatres.com/movie-theatres/seattle-tacoma/amc-alderwood-mall-16/showtimes")